# Overwatch RL Lab: learning curves from real game logs

Every number here comes from log files that Overwatch wrote while the Workshop mode ran
(`data/logs/*.txt`). Nothing is simulated. To add your own session, copy the newest file from
`Documents\Overwatch\Workshop\` into `data/logs/` and run all cells again.

Column meanings: `docs/logging.md`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "logs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "analysis"))

import pandas as pd
from rllab.logs import parse_log, episodes_frame, EXPERIMENTS
from rllab.plots import summary_table, learning_curves
from rllab.replay import replay

files = sorted((ROOT / "data" / "logs").glob("*.txt"))
log = parse_log(*files)
episodes = episodes_frame(log)
print(f"{len(files)} log files, {len(log.ep)} episodes, {len(log.step)} logged steps")
assert not log.skipped, log.skipped[:3]

8 log files, 344 episodes, 7594 logged steps


## 1. What was measured about the arena

One `CAL` line per run: the island's edges as found by ray casts, and Tracer's timed run speed.

In [2]:
pd.DataFrame(log.cal)[["file", "x_min", "x_max", "z_min", "z_max", "floor_y", "cell", "run_speed", "coast_dist", "act_duration"]].drop_duplicates()

,file,x_min,x_max,z_min,z_max,floor_y,cell,run_speed,coast_dist,act_duration
0,2026-11-21_blink_fixed_duration_qlearning.txt,-20,20,-20,20,0,2.0,5.98,0.25,0.33
1,2026-11-21_cliffwalk_qlearning_then_controls.txt,-20,20,-20,20,0,2.0,5.98,0.25,0.33
4,2026-11-21_cliffwalk_sarsa.txt,-20,20,-20,20,0,2.0,5.98,0.25,0.33
5,2026-11-21_demo_patrol.txt,-20,20,-20,20,0,2.0,5.98,0.25,0.33
6,2026-11-21_nondefault_settings_qlearning.txt,-20,20,-20,20,0,2.5,8.97,0.37,0.28
7,2026-11-21_nondefault_widowmaker_spectator_hos...,-20,20,-20,20,0,2.0,5.98,0.25,0.33
8,2026-11-21_standard_build_smoke.txt,-20,20,-20,20,0,2.0,5.98,0.25,0.33
10,2026-11-21_widowmaker_sarsa.txt,-20,20,-20,20,0,2.0,5.98,0.25,0.33


## 2. Did the game compute the updates correctly?

Where per-step logging was on, the replay applies the reference update
(`analysis/rllab/reference.py`) to the transitions the game logged and compares every target and
every Q-value with what the game wrote. A mismatch would be listed here.

In [3]:
report = replay(log)
print(f"steps checked: {report.steps_checked}, updates checked: {report.updates_checked}, episodes re-added: {report.episodes_checked}")
print(dict(report.by_algo))
if report.training_steps:
    print(f"exploratory draws: {report.explore_steps / report.training_steps:.1%} of training steps")
print("mismatches:", report.mismatches[:5] if report.mismatches else "none")

steps checked: 7433, updates checked: 6540, episodes re-added: 334
{'Q-learning': 3177, 'SARSA': 3363}
exploratory draws: 10.2% of training steps
mismatches: none


## 3. Episodes per run

One row per file, run, algorithm and mode. `fall_rate` and `goal_rate` are shares of episodes.
`mean_edge_dist` is Tracer's average distance from the island edge in metres (row 0 is centred
1.5 m from the edge, row 1 at 3.5 m).

In [4]:
runs = pd.DataFrame(log.run)
runs["experiment_name"] = runs["experiment"].map(EXPERIMENTS)
learning = episodes.merge(runs[["file", "run", "experiment_name", "closed_loop", "use_blink"]], on=["file", "run"])
summary_table(learning).round(2)

,file,run,algo_name,mode_name,episodes,mean_return,goal_rate,fall_rate,mean_steps,mean_edge_dist,epsilon,alpha,gamma
0,2026-11-21_blink_fixed_duration_qlearning.txt,1,Q-learning,training,29,-18.72,0.59,0.41,6.66,2.05,0.10,0.5,1.0
1,2026-11-21_cliffwalk_qlearning_then_controls.txt,1,Q-learning,training,119,-15.76,0.71,0.28,23.32,3.06,0.10,0.5,1.0
2,2026-11-21_cliffwalk_qlearning_then_controls.txt,1,Q-learning,evaluation,12,37.00,1.00,0.00,13.00,2.39,0.10,0.5,1.0
3,2026-11-21_cliffwalk_qlearning_then_controls.txt,2,SARSA,evaluation,3,-132.00,0.00,1.00,32.00,3.42,0.10,0.5,1.0
4,2026-11-21_cliffwalk_qlearning_then_controls.txt,3,SARSA,evaluation,1,-101.00,0.00,1.00,1.00,0.00,0.10,0.5,1.0
5,2026-11-21_cliffwalk_sarsa.txt,1,SARSA,training,115,-3.17,0.81,0.19,24.47,3.91,0.10,0.5,1.0
6,2026-11-21_cliffwalk_sarsa.txt,1,SARSA,evaluation,3,-150.00,0.00,0.00,100.00,3.43,0.10,0.5,1.0
7,2026-11-21_demo_patrol.txt,1,SARSA,evaluation,20,33.00,1.00,0.00,17.00,6.45,0.10,0.5,1.0
8,2026-11-21_nondefault_settings_qlearning.txt,1,Q-learning,training,10,-93.80,0.00,0.60,20.90,3.49,0.15,0.3,0.9
9,2026-11-21_nondefault_widowmaker_spectator_hos...,1,SARSA,training,11,-121.45,0.00,0.73,22.36,2.92,0.10,0.5,1.0


## 4. Learning curves (training episodes, standard cliff walk only)

Top: return per episode. Middle: share of episodes that ended in a fall. Bottom: mean distance
from the edge. Every line is smoothed over 20 episodes. A thin line is one run; where an algorithm
has several runs, the thick line is their mean.

Only runs made under the standard settings are drawn (cold start, closed-loop moves, four actions,
default rewards, epsilon 0.1, alpha 0.5, gamma 1). Runs with other settings are a different
experiment and would not be comparable.

In [5]:
from rllab.compare import standard_runs, late_training_table, by_algorithm
from rllab.sanity import too_fast_episodes, duplicate_matches

assert not too_fast_episodes(log), "some episodes are too fast to be real play"
assert not duplicate_matches(log), "a match is in data/logs twice"

keep = pd.DataFrame(standard_runs(log), columns=["file", "run"])
cliff = learning.merge(keep, on=["file", "run"])
cliff = cliff[cliff["mode_name"] == "training"]
if len(cliff):
    figure = learning_curves(cliff, window=20)
    figure.savefig(ROOT / "analysis" / "figures" / "learning_curves.png", dpi=160)
    display(figure)
else:
    print("No standard cliff-walk training episodes in data/logs yet.")

<Figure size 800x900 with 3 Axes>

## 5. SARSA against Q-learning, run by run

One row per full run (at least 100 training episodes). The `late_` columns cover the last 40
training episodes, when both agents are still exploring. This is the comparison Lab 02 asks about.
The `eval_` columns are the greedy episodes run straight afterwards, with exploration off.

In [6]:
rows = late_training_table(log)
runs_table = pd.DataFrame(rows)
runs_table[["algo", "file", "run", "training_episodes", "minutes", "first_goal_episode", "late_mean_return",
            "late_falls", "late_fall_rate", "late_mean_edge_dist", "late_median_goal_steps",
            "eval_n", "eval_goals", "eval_timeouts", "eval_falls", "eval_median_goal_steps"]].sort_values(["algo", "file", "run"]).round(2)

,algo,file,run,training_episodes,minutes,first_goal_episode,late_mean_return,late_falls,late_fall_rate,late_mean_edge_dist,late_median_goal_steps,eval_n,eval_goals,eval_timeouts,eval_falls,eval_median_goal_steps
0,Q-learning,2026-11-21_cliffwalk_qlearning_then_controls.txt,1,119,20.9,13,-1.3,10,0.25,2.40,15.0,12,12,0,0,13.0
1,SARSA,2026-11-21_cliffwalk_sarsa.txt,1,115,22.8,12,18.2,4,0.10,3.88,17.0,3,0,3,0,NaN


## 6. The two algorithms compared at the level of runs

Each run is one sample, so the test below works on run means, not on episodes. It is an exact
permutation test: of all the ways to deal these runs into two groups, the share that gives a gap
at least as large as the one seen. With three runs against three, the smallest value it can ever
return is 0.10, so read it as "as separated as three runs each can show", not as proof.

In [7]:
summary = by_algorithm(rows)
for algo, stats in sorted(summary["algorithms"].items()):
    print(f"{algo}: {stats['runs']} runs, falls in late training {stats['pooled_late_falls']} of {stats['pooled_late_n']} episodes")
    for name in ("late_mean_return", "late_fall_rate", "late_mean_edge_dist"):
        values = stats[name]
        print(f"    {name[5:]:<15} runs: {[round(v, 2) for v in values['values']]}   mean {values['mean']:.2f}")
for name, p in summary["tests"].items():
    print(f"exact permutation test on run means, {name[5:]}: p = {p:.2f}")

Q-learning: 1 runs, falls in late training 10 of 40 episodes
    mean_return     runs: [-1.3]   mean -1.30
    fall_rate       runs: [0.25]   mean 0.25
    mean_edge_dist  runs: [2.4]   mean 2.40
SARSA: 1 runs, falls in late training 4 of 40 episodes
    mean_return     runs: [18.2]   mean 18.20
    fall_rate       runs: [0.1]   mean 0.10
    mean_edge_dist  runs: [3.88]   mean 3.88


## 7. Everything else that was recorded

Runs under other settings (Widowmaker, Blink, non-default values, the scripted patrol), by mode.
These are checks that the features work, not experiments.

In [8]:
other = learning.merge(keep.assign(standard=True), on=["file", "run"], how="left")
other = other[other["standard"].isna()]
other.groupby(["experiment_name", "algo_name", "mode_name", "file", "run"]).agg(
    episodes=("ret", "size"), mean_return=("ret", "mean"), goal_rate=("goal", "mean"),
    fall_rate=("fell", "mean"), mean_steps=("steps", "mean"), hits=("hits", "sum"),
).round(2)

episodes  \
experiment_name         algo_name  mode_name  file                                               run             
arena demo              SARSA      evaluation 2026-11-21_demo_patrol.txt                         1          20   
cliff walk              Q-learning training   2026-11-21_blink_fixed_duration_qlearning.txt      1          29   
                                              2026-11-21_nondefault_settings_qlearning.txt       1          10   
cliff walk + Widowmaker SARSA      training   2026-11-21_nondefault_widowmaker_spectator_host... 1          11   
                                              2026-11-21_widowmaker_sarsa.txt                    1          11   

                                                                                                      mean_return  \
experiment_name         algo_name  mode_name  file                                               run                
arena demo              SARSA      evaluation 2026-11-21_demo_patrol.txt                         1          33.00   
cliff walk              Q-learning training   2026-11-21_blink_fixed_duration_qlearning.txt      1         -18.72   
                                              2026-11-21_nondefault_settings_qlearning.txt       1         -93.80   
cliff walk + Widowmaker SARSA      training   2026-11-21_nondefault_widowmaker_spectator_host... 1        -121.45   
                                              2026-11-21_widowmaker_sarsa.txt                    1        -127.55   

                                                                                                      goal_rate  \
experiment_name         algo_name  mode_name  file                                               run              
arena demo              SARSA      evaluation 2026-11-21_demo_patrol.txt                         1         1.00   
cliff walk              Q-learning training   2026-11-21_blink_fixed_duration_qlearning.txt      1         0.59   
                                              2026-11-21_nondefault_settings_qlearning.txt       1         0.00   
cliff walk + Widowmaker SARSA      training   2026-11-21_nondefault_widowmaker_spectator_host... 1         0.00   
                                              2026-11-21_widowmaker_sarsa.txt                    1         0.00   

                                                                                                      fall_rate  \
experiment_name         algo_name  mode_name  file                                               run              
arena demo              SARSA      evaluation 2026-11-21_demo_patrol.txt                         1         0.00   
cliff walk              Q-learning training   2026-11-21_blink_fixed_duration_qlearning.txt      1         0.41   
                                              2026-11-21_nondefault_settings_qlearning.txt       1         0.60   
cliff walk + Widowmaker SARSA      training   2026-11-21_nondefault_widowmaker_spectator_host... 1         0.73   
                                              2026-11-21_widowmaker_sarsa.txt                    1         0.64   

                                                                                                      mean_steps  \
experiment_name         algo_name  mode_name  file                                               run               
arena demo              SARSA      evaluation 2026-11-21_demo_patrol.txt                         1         17.00   
cliff walk              Q-learning training   2026-11-21_blink_fixed_duration_qlearning.txt      1          6.66   
                                              2026-11-21_nondefault_settings_qlearning.txt       1         20.90   
cliff walk + Widowmaker SARSA      training   2026-11-21_nondefault_widowmaker_spectator_host... 1         22.36   
                                              2026-11-21_widowmaker_sarsa.txt                    1         27.55   

                                                          